# USGS Study Package From Primitives

This notebook shows the preferred pattern for USGS gauge study assembly in `ras-commander`.

- Use composable primitives from `ras_commander.usgs`
- Keep study packaging, readiness assessment, and report assembly in notebook or project code
- Avoid treating workflow wrappers as the core library contract


In [ ]:
from pathlib import Path
import json

from ras_commander.usgs import (
    UsgsObservations,
    UsgsDrainageAreaComparison,
)


In [ ]:
site_id = "05577500"
start_datetime = "2024-01-01"
end_datetime = "2024-12-31"

metadata = UsgsObservations.get_gauge_metadata(site_id)
continuous_flow = UsgsObservations.get_dataset(
    site_id=site_id,
    dataset_id="continuous_flow",
    start_datetime=start_datetime,
    end_datetime=end_datetime,
)
daily_flow = UsgsObservations.get_dataset(
    site_id=site_id,
    dataset_id="daily_flow",
    start_datetime=start_datetime,
    end_datetime=end_datetime,
)

continuous_flow_summary = UsgsObservations.summarize_dataset(
    dataset_id="continuous_flow",
    frame=continuous_flow,
    status="ok" if not continuous_flow.empty else "empty",
)
continuous_flow_gaps = UsgsObservations.analyze_gaps(
    dataset_id="continuous_flow",
    frame=continuous_flow,
)

metadata, continuous_flow_summary, continuous_flow_gaps


In [ ]:
# Notebook-local drainage-area review.
# Replace the example values below with official basin, TauDEM, and model areas.
drainage_area_review = UsgsDrainageAreaComparison.compare_areas(
    gauge_area_sqmi=metadata.get("drainage_area_sqmi"),
    official_basin_area_sqmi=125.8,
    taudem_area_sqmi=124.9,
    model_area_sqmi=126.4,
)
drainage_area_review


In [ ]:
# Notebook-local report assembly.
# This is intentionally local workflow code instead of core package API.
report = {
    "study": {
        "site_id": site_id,
        "start_datetime": start_datetime,
        "end_datetime": end_datetime,
    },
    "gauge_metadata": metadata,
    "datasets": {
        "continuous_flow": continuous_flow_summary,
        "daily_flow": UsgsObservations.summarize_dataset(
            dataset_id="daily_flow",
            frame=daily_flow,
            status="ok" if not daily_flow.empty else "empty",
        ),
    },
    "data_gaps": {
        "continuous_flow": continuous_flow_gaps,
        "daily_flow": UsgsObservations.analyze_gaps(
            dataset_id="daily_flow",
            frame=daily_flow,
        ),
    },
    "drainage_area_review": drainage_area_review,
}

output_dir = Path("working") / f"USGS_{site_id}_study_package"
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / "report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
report_path
